<a href="https://colab.research.google.com/github/casper-justus/swahili-gpt/blob/main/MiniGPT_Kiswahili_Resumable_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kiswahili MiniGPT: Multi-Day Resumable Training
This notebook trains a ~30 Million parameter GPT model on a pure Kiswahili dataset.
It includes Google Drive integration to save checkpoints so you can resume training over multiple days without losing progress when Colab disconnects.

**Before running:**
1. Connect to **T4 GPU** (Runtime > Change runtime type).



In [1]:
# Cell 1: Mount Google Drive (For saving checkpoints securely)
from google.colab import drive
drive.mount('/content/drive')

import os
# Checkpoints will be saved here in your Google Drive
CKPT_DIR = "/content/drive/MyDrive/MiniGPT_Kiswahili_Checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Checkpoints will be stored at: {CKPT_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoints will be stored at: /content/drive/MyDrive/MiniGPT_Kiswahili_Checkpoints


In [2]:
# Cell 2: Install required libraries (Explicitly matching CUDA 12 for Colab)
!pip uninstall -y jax jaxlib flax optax orbax-checkpoint
!pip install "jax[cuda12]" flax optax orbax-checkpoint datasets tokenizers


Found existing installation: jax 0.10.0
Uninstalling jax-0.10.0:
  Successfully uninstalled jax-0.10.0
Found existing installation: jaxlib 0.10.0
Uninstalling jaxlib-0.10.0:
  Successfully uninstalled jaxlib-0.10.0
Found existing installation: flax 0.12.7
Uninstalling flax-0.12.7:
  Successfully uninstalled flax-0.12.7
Found existing installation: optax 0.2.8
Uninstalling optax-0.2.8:
  Successfully uninstalled optax-0.2.8
Found existing installation: orbax-checkpoint 0.11.37
Uninstalling orbax-checkpoint-0.11.37:
  Successfully uninstalled orbax-checkpoint-0.11.37
  Using cached flax-0.12.7-py3-none-any.whl.metadata (11 kB)
  Using cached optax-0.2.8-py3-none-any.whl.metadata (7.9 kB)
  Using cached orbax_checkpoint-0.11.37-py3-none-any.whl.metadata (2.8 kB)
  Using cached jax-0.10.0-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.10.0-cp312-cp312-manylinux_2_27_x86_64.whl.metadata (1.3 kB)
Using cached flax-0.12.7-py3-none-any.whl (525 kB)
Using cached optax-0.2.8-py3-none-

In [3]:
# Cell 3: Imports and Hyperparameters (~30M Parameters)
import jax
import jax.numpy as jnp
from flax import nnx
import optax
import orbax.checkpoint as ocp
from datasets import load_dataset
from tokenizers import Tokenizer

SEQ_LEN = 1024          # Context window
BATCH_SIZE = 4       # Fits nicely in T4 16GB VRAM
EMB_SIZE = 512         # Embedding dimension
NUM_HEADS = 8          # Attention heads
NUM_LAYERS = 6         # Transformer blocks
LEARNING_RATE = 3e-4
TOTAL_STEPS = 200000    # Total training steps over multiple days
SAVE_EVERY = 5000       # Backup to Google Drive every 500 steps


In [4]:
# Cell 4: Load Kiswahili Dataset & Tokenizer
print("Loading Tokenizer...")
# Ensure kenya_tokenizer.json is uploaded!
tokenizer = Tokenizer.from_file("/content/drive/MyDrive/MiniGPT_Kiswahili_Checkpoints/kenyan_tokenizer.json")
VOCAB_SIZE = tokenizer.get_vocab_size()
print(f"Tokenizer loaded! Vocabulary size: {VOCAB_SIZE}")

print("Loading Kiswahili Dataset...")
# We use a modern Parquet-based Kiswahili text corpus to avoid script errors
dataset = load_dataset("marcoharuni95/swahili-text-corpus", split="train")
print(f"Dataset loaded! Total examples: {len(dataset)}")

def create_data_generator(dataset, tokenizer, batch_size, seq_len):
    all_tokens = []
    print("Tokenizing Kiswahili data...")
    # Fetch a large chunk of the text for training
    for text in dataset['text'][:200000]:
        if text:
            tokens = tokenizer.encode(text).ids
            all_tokens.extend(tokens)

    print(f"Total Kiswahili tokens processed: {len(all_tokens)}")
    i = 0
    while True:
        batch_x, batch_y = [], []
        for _ in range(batch_size):
            if i + seq_len + 1 >= len(all_tokens):
                i = 0
            chunk = all_tokens[i : i + seq_len + 1]
            batch_x.append(chunk[:-1])
            batch_y.append(chunk[1:])
            i += seq_len
        yield jnp.array(batch_x), jnp.array(batch_y)

dataloader = create_data_generator(dataset, tokenizer, BATCH_SIZE, SEQ_LEN)


Loading Tokenizer...
Tokenizer loaded! Vocabulary size: 10000
Loading Kiswahili Dataset...
Dataset loaded! Total examples: 46708


In [5]:
# Cell 5: Architecture (with NNX Sequential fix)
class Block(nnx.Module):
    def __init__(self, emb_size, num_heads, rngs):
        self.ln_1 = nnx.LayerNorm(emb_size, rngs=rngs)
        self.attn = nnx.MultiHeadAttention(num_heads=num_heads, in_features=emb_size, decode=False, rngs=rngs)
        self.ln_2 = nnx.LayerNorm(emb_size, rngs=rngs)
        self.mlp = nnx.Sequential(
            nnx.Linear(emb_size, 4 * emb_size, rngs=rngs),
            nnx.gelu,
            nnx.Linear(4 * emb_size, emb_size, rngs=rngs)
        )

    def __call__(self, x, mask):
        x = x + self.attn(self.ln_1(x), mask=mask)
        x = x + self.mlp(self.ln_2(x))
        return x

class MiniGPT(nnx.Module):
    def __init__(self, vocab_size, seq_len, emb_size, num_heads, num_layers, rngs):
        self.token_emb = nnx.Embed(vocab_size, emb_size, rngs=rngs)
        self.pos_emb = nnx.Embed(seq_len, emb_size, rngs=rngs)

        # Wrapped in nnx.Sequential to avoid Pytree list errors
        self.blocks = nnx.Sequential(*[Block(emb_size, num_heads, rngs) for _ in range(num_layers)])

        self.ln_f = nnx.LayerNorm(emb_size, rngs=rngs)
        self.lm_head = nnx.Linear(emb_size, vocab_size, rngs=rngs)

    def __call__(self, idx):
        b, t = idx.shape
        pos = jnp.arange(0, t, dtype=jnp.int32)[None, :]
        x = self.token_emb(idx) + self.pos_emb(pos)
        mask = nnx.make_causal_mask(jnp.ones((b, t)))

        for block in self.blocks.layers:
            x = block(x, mask)

        return self.lm_head(self.ln_f(x))


In [6]:
# Cell 6: Setup Training (with Flax 0.11+ fixes)
rngs = nnx.Rngs(0)
model = MiniGPT(VOCAB_SIZE, SEQ_LEN, EMB_SIZE, NUM_HEADS, NUM_LAYERS, rngs)

tx = optax.adamw(learning_rate=LEARNING_RATE)

# Fix: explicitly provide wrt=nnx.Param
optimizer = nnx.Optimizer(model, tx, wrt=nnx.Param)

@nnx.jit
def train_step(model, optimizer, batch_x, batch_y):
    def loss_fn(model):
        logits = model(batch_x)
        return optax.softmax_cross_entropy_with_integer_labels(logits, batch_y).mean()

    loss, grads = nnx.value_and_grad(loss_fn)(model)

    # Fix: Provide both model and grads to update
    optimizer.update(model, grads)
    return loss


In [7]:
# Cell 7: Resume / Restore Checkpoint Logic
options = ocp.CheckpointManagerOptions(max_to_keep=3, create=True)
mngr = ocp.CheckpointManager(CKPT_DIR, options=options)

# FIX: Split the model and optimizer separately to prevent formatting errors
_, model_state = nnx.split(model)
_, opt_state = nnx.split(optimizer)

state_tree = {'model': model_state, 'opt': opt_state}

start_step = 0

if mngr.latest_step() is not None:
    start_step = mngr.latest_step()
    print(f"Found existing checkpoint! Resuming from step {start_step}...")

    restored = mngr.restore(start_step, args=ocp.args.StandardRestore(state_tree))
    nnx.update(model, restored['model'])
    nnx.update(optimizer, restored['opt'])
    print("Model and Optimizer weights fully restored.")
else:
    print("No checkpoint found. Starting fresh from Step 0.")

No checkpoint found. Starting fresh from Step 0.


In [ ]:
# Cell 8: The Training Loop
print("Starting Training...")

for step in range(start_step, TOTAL_STEPS):
    batch_x, batch_y = next(dataloader)

    loss = train_step(model, optimizer, batch_x, batch_y)

    if step % 50 == 0:
        print(f"Step {step:05d} | Loss: {loss:.4f}")

    # Save checkpoint to Google Drive
    if step > 0 and step % SAVE_EVERY == 0:

        # FIX: Split the model and optimizer separately before saving
        _, current_model_state = nnx.split(model)
        _, current_opt_state = nnx.split(optimizer)

        current_tree = {'model': current_model_state, 'opt': current_opt_state}

        mngr.save(step, args=ocp.args.StandardSave(current_tree))
        print(f"--> Checkpoint saved securely to Google Drive at step {step}")

print("Training phase complete!")

Starting Training...
Tokenizing Kiswahili data...
Total Kiswahili tokens processed: 17633324
Step 00000 | Loss: 9.5638
Step 00050 | Loss: 7.3661
Step 00100 | Loss: 6.9166
Step 00150 | Loss: 7.0180
Step 00200 | Loss: 6.2425
Step 00250 | Loss: 5.8596
Step 00300 | Loss: 6.2795
Step 00350 | Loss: 5.7340
Step 00400 | Loss: 5.8195
Step 00450 | Loss: 5.8937
Step 00500 | Loss: 5.9723
Step 00550 | Loss: 5.8526
Step 00600 | Loss: 5.7223
Step 00650 | Loss: 5.5014
Step 00700 | Loss: 5.4276
Step 00750 | Loss: 5.5952
Step 00800 | Loss: 5.9681
Step 00850 | Loss: 5.5368
Step 00900 | Loss: 5.1657
Step 00950 | Loss: 5.3737
Step 01000 | Loss: 5.2623
Step 01050 | Loss: 4.8569
Step 01100 | Loss: 5.1869
Step 01150 | Loss: 5.0105
Step 01200 | Loss: 5.1934
Step 01250 | Loss: 5.6007
Step 01300 | Loss: 5.7764
Step 01350 | Loss: 5.0218
Step 01400 | Loss: 5.5764
Step 01450 | Loss: 5.0323
Step 01500 | Loss: 4.8584
Step 01550 | Loss: 5.7788
Step 01600 | Loss: 5.1223
Step 01650 | Loss: 4.9224
Step 01700 | Loss: 5.12